### 05. Exportação para o Power BI (esquema estrela)

**Etapa CRISP-DM:** implantação (deploy).

Gera em `data/processed/bi/` as tabelas do modelo dimensional (Kimball e Ross, 2013), prontas para importar no Power BI:

| Tabela | Tipo | Grão (uma linha por) |
|---|---|---|
| `fato_vendas` | fato | item válido da cesta |
| `fato_resgate` | fato | cupom resgatado |
| `fato_exposicao` | fato | domicílio x campanha recebida |
| `fato_domicilio_corte` | fato | domicílio x corte, com RFM, rótulo e score |
| `dim_domicilio` | dimensão | domicílio |
| `dim_produto` | dimensão | produto |
| `dim_campanha` | dimensão | campanha |
| `dim_semana` | dimensão | semana |
| `resultado_hipoteses`, `resultado_modelos` | apoio | resultado dos notebooks 03 e 04 |

Também grava `reports/bi/conferencia_bi.json` com os totais que o Power BI precisa reproduzir.

#### 0. Bibliotecas e pastas

In [1]:
import json
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PASTA_INTERIM = RAIZ / "data" / "interim"
PASTA_PROC = RAIZ / "data" / "processed"
PASTA_BI = PASTA_PROC / "bi"
PASTA_REP = RAIZ / "reports" / "bi"
for pasta in [PASTA_BI, PASTA_REP]:
    pasta.mkdir(parents=True, exist_ok=True)

#### 1. Fatos de vendas, resgates e exposição

In [2]:
transacoes = pd.read_csv(PASTA_INTERIM / "transacoes.csv")
validas = transacoes[~transacoes["flag_valor_invalido"]]   
fato_vendas = validas[["household_key", "basket_id", "day", "week_no", "product_id", "store_id",
                       "quantity", "sales_value", "retail_disc", "coupon_disc"]]
fato_vendas.to_csv(PASTA_BI / "fato_vendas.csv", index=False)

resgates = pd.read_csv(PASTA_INTERIM / "cupons_resgatados.csv")
resgates["week_no"] = (resgates["day"] + 1) // 7 + 1        
resgates.to_csv(PASTA_BI / "fato_resgate.csv", index=False)

exposicao = pd.read_csv(PASTA_INTERIM / "campanha_domicilio.csv")[["household_key", "campaign"]]
exposicao.to_csv(PASTA_BI / "fato_exposicao.csv", index=False)
print(len(fato_vendas), len(resgates), len(exposicao))

2576815 2318 7208


#### 2. Fato domicílio x corte (RFM, rótulo e score)

In [3]:
base = pd.read_csv(PASTA_PROC / "base_modelagem.csv")
score = pd.read_csv(PASTA_PROC / "score_risco.csv")[["household_key", "score_risco", "faixa_risco", "prioridade", "rank_prioridade"]]
score["corte"] = "score"
fato_dc = base.drop(columns=["renda", "tamanho_domicilio", "tem_demografia"]).merge(score, on=["corte", "household_key"], how="left")
fato_dc.to_csv(PASTA_BI / "fato_domicilio_corte.csv", index=False)
fato_dc.groupby("corte").size()

corte
score     2432
teste     2417
treino    2409
dtype: int64

#### 3. Dimensões

In [4]:
# domicílio: todos os 2.500, com demografia quando existir
demografia = pd.read_csv(PASTA_INTERIM / "demografia.csv")
dim_dom = pd.DataFrame({"household_key": sorted(transacoes["household_key"].unique())})
dim_dom = dim_dom.merge(demografia, on="household_key", how="left")
dim_dom["tem_demografia"] = dim_dom["income_desc"].notna().map({True: "sim", False: "não"})
dim_dom = dim_dom.rename(columns={"age_desc": "faixa_idade", "income_desc": "renda", "household_size_desc": "tamanho_domicilio",
                                  "hh_comp_desc": "composicao", "homeowner_desc": "moradia"})
dim_dom = dim_dom[["household_key", "tem_demografia", "faixa_idade", "renda", "tamanho_domicilio", "composicao", "moradia"]]
dim_dom.to_csv(PASTA_BI / "dim_domicilio.csv", index=False)

produtos = pd.read_csv(PASTA_INTERIM / "produtos.csv")
produtos[["product_id", "department", "brand", "commodity_desc", "sub_commodity_desc"]].to_csv(PASTA_BI / "dim_produto.csv", index=False)

campanhas = pd.read_csv(PASTA_INTERIM / "campanhas.csv")
campanhas["semana_inicio"] = (campanhas["start_day"] + 1) // 7 + 1
campanhas["semana_fim"] = (campanhas["end_day"] + 1) // 7 + 1
campanhas.rename(columns={"description": "tipo"}).to_csv(PASTA_BI / "dim_campanha.csv", index=False)

# semana: dia inicial e final e o corte a que pertence a observação
semanas = transacoes.groupby("week_no")["day"].agg(dia_inicio="min", dia_fim="max").reset_index()
semanas["periodo"] = pd.cut(semanas["week_no"], bins=[0, 18, 32, 102],
                            labels=["entrada gradual (1-18)", "antes das campanhas (19-32)", "com campanhas (33-102)"])
semanas.to_csv(PASTA_BI / "dim_semana.csv", index=False)
print(len(dim_dom), len(produtos), len(campanhas), len(semanas))

2500 92353 30 102


#### 4. Resultados dos notebooks 03 e 04 (para os painéis)

In [5]:
sens = pd.read_csv(RAIZ / "reports" / "hipoteses" / "sensibilidade.csv")
sens["significativo_5pct"] = (sens["p_valor"] < 0.05).map({True: "sim", False: "não"})
sens.to_csv(PASTA_BI / "resultado_hipoteses.csv", index=False)
met = pd.read_csv(RAIZ / "reports" / "modelo" / "metricas_teste.csv")
met.to_csv(PASTA_BI / "resultado_modelos.csv", index=False)
met

,modelo,AUC_treino,AUC_teste,acuracia,precisao,recall,F1,precisao_top10,lift_top10
0,regra_recencia,0.853,0.815,0.873,0.395,0.488,0.437,0.424,4.205
1,logistica_rfm,0.892,0.862,0.876,0.405,0.492,0.444,0.421,4.175
2,logistica_completa,0.894,0.857,0.878,0.409,0.471,0.438,0.430,4.257


#### 5. Totais de conferência
Valores que as medidas DAX precisam reproduzir. Se o Power BI mostrar outro número, há erro de relacionamento ou de filtro.

In [6]:
def taxa(df, col="inativo_8s"):
    s = df[col].dropna()
    return round(100 * s.mean(), 1)

conf = {
    "receita_itens_validos": round(float(fato_vendas["sales_value"].sum()), 2),
    "cestas": int(fato_vendas["basket_id"].nunique()),
    "domicilios_com_venda": int(fato_vendas["household_key"].nunique()),
    "ticket_medio_geral": round(float(fato_vendas["sales_value"].sum() / fato_vendas["basket_id"].nunique()), 2),
}
for corte, g in fato_dc.groupby("corte"):
    conf[corte] = {
        "domicilios": len(g),
        "recencia_media_dias": round(float(g["recencia_dias"].mean()), 2),
        "frequencia_media": round(float(g["frequencia"].mean()), 2),
        "ticket_medio_domicilio": round(float(g["ticket_medio"].mean()), 2),
        "taxa_resgate_entre_expostos_pct": round(100 * g.loc[g["recebeu_campanha"] == 1, "resgatou_cupom"].mean(), 1),
    }
    if corte != "score":
        conf[corte]["taxa_inativacao_8s_pct"] = taxa(g)
        conf[corte]["taxa_inativacao_com_campanha_pct"] = taxa(g[g["recebeu_campanha"] == 1])
        conf[corte]["taxa_inativacao_sem_campanha_pct"] = taxa(g[g["recebeu_campanha"] == 0])
    else:
        conf[corte]["receita_esperada_em_risco"] = round(float(g["prioridade"].sum()), 2)
        conf[corte]["domicilios_risco_alto"] = int((g["faixa_risco"] == "alto").sum())
with open(PASTA_REP / "conferencia_bi.json", "w", encoding="utf-8") as f:
    json.dump(conf, f, ensure_ascii=False, indent=2)
conf

{'receita_itens_validos': 8057443.8,
 'cestas': 275539,
 'domicilios_com_venda': 2500,
 'ticket_medio_geral': 29.24,
 'score': {'domicilios': 2432,
  'recencia_media_dias': 18.05,
  'frequencia_media': 31.04,
  'ticket_medio_domicilio': 33.29,
  'taxa_resgate_entre_expostos_pct': np.float64(20.9),
  'receita_esperada_em_risco': 75298.65,
  'domicilios_risco_alto': 244},
 'teste': {'domicilios': 2417,
  'recencia_media_dias': 15.88,
  'frequencia_media': 32.01,
  'ticket_medio_domicilio': 32.36,
  'taxa_resgate_entre_expostos_pct': np.float64(24.3),
  'taxa_inativacao_8s_pct': np.float64(10.1),
  'taxa_inativacao_com_campanha_pct': np.float64(4.0),
  'taxa_inativacao_sem_campanha_pct': np.float64(18.4)},
 'treino': {'domicilios': 2409,
  'recencia_media_dias': 16.45,
  'frequencia_media': 32.54,
  'ticket_medio_domicilio': 31.56,
  'taxa_resgate_entre_expostos_pct': np.float64(21.4),
  'taxa_inativacao_8s_pct': np.float64(8.8),
  'taxa_inativacao_com_campanha_pct': np.float64(2.8),
  't

In [7]:
for p in sorted(PASTA_BI.glob("*.csv")):
    print(f"{p.name:28s} {p.stat().st_size / 1e6:7.1f} MB")

dim_campanha.csv                 0.0 MB
dim_domicilio.csv                0.1 MB
dim_produto.csv                  5.7 MB
dim_semana.csv                   0.0 MB
fato_domicilio_corte.csv         0.7 MB
fato_exposicao.csv               0.1 MB
fato_resgate.csv                 0.1 MB
fato_vendas.csv                133.9 MB
resultado_hipoteses.csv          0.0 MB
resultado_modelos.csv            0.0 MB
